# Final Project: Trending Topic Detection in Twitter
Mónica Martín 

Marcos Gastón

Emma Rodríguez


# Table of Contents
- [Introduction](#Introduction)
- [Preprocessing](#Preprocessing)
    - [Fetching Data](#Preprocessing-Part-1-Fetching-Data)
    - [Pipeline](#Preprocessing-Part-2-The-Pipeline)
    - [Language detection](#Filtering-language-and-tokens-in-English)
- [Time Window](#The-Time-Window)
- [Ngram Generation](#Generating-Ngrams)
- [Frequency Penalization](#Frequency-Penalization)
- [Boosting](#Boosting)
- [Clustering](#Clustering)
    - [Hierarchical Clustering](#Hierarchical-Clustering)
- [Summary Function](#Summary-Function)
- [Conclusion](#Conclusion)

#Introduction

###Bngrams
To identify trending topics on Twitter, we used n-grams, a method that captures common word patterns like phrases or terms that frequently appear together. We found it more intuitive then other approches as well as more interesting as wethought it would work well on Twitter, where many posts are retweets or contain repeated phrases, making important n-grams easy to spot.

After arduous preprocessing we also tracked how word usage changed over time, highlighting new and emerging trends while penalizing older, ongoing topics. Proper nouns, such as names of people, places, or organizations, were given extra importance since they were central to many discussions. Finally, we grouped related n-grams into clusters to create clear and cohesive topics.

This method was simple, efficient, and effective for capturing the fast-moving trends on Twitter.

The work was long but we got very satisfying results.

##Preprocessing
In the following chunks we spent a lot of time on preprocessing. In order to organize our data obtained from the link into something structured, extract necessary ordered information and removing redunant data we employed filtering, cleaning, removing stopwords, detecting language... 
Many different functions we created were executed orderly in a large `pipeline` with the finality of simplifying ou future activity, what's really important in this project.


In [0]:
# Imports
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, window, collect_list, explode, regexp_replace, flatten, udf, when, array_contains, lit, split
from pyspark.ml.feature import NGram
import requests
from pyspark.sql.types import BooleanType
import pyspark.sql.functions as F
import re
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import Tokenizer, Normalizer, LanguageDetectorDL, StopWordsCleaner
from pyspark.ml import Pipeline
from sparknlp.annotator import NerDLModel, NerConverter
from sparknlp.annotator import WordEmbeddingsModel
from pyspark.sql.window import Window
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from IPython.display import display

###Preprocessing Part 1 (Fetching Data)
This is a function to obtain the data and open ourSpark session. If we wanted to change the dataset we use we would change the link defined as `"datafile"`.

In [0]:
def fetch_and_process_data(start_year, start_month, start_day, start_hour, start_minute, duration_minutes):
    # Fetch the data
    datafile = f"http://mcomputing.tsc.uc3m.es/get_tweets.php?start_month={start_month}&start_day={start_day}&start_hour={start_hour}&start_minute={start_minute}&minutes_length={duration_minutes}"
    response = requests.get(datafile)
    
    # Save the file temporarily 
    temp_local_path = "/tmp/temporal.json"
    with open(temp_local_path, "w", encoding="utf-8") as f:
        f.write(response.text)

    # Start Spark Session
    spark = SparkSession.builder \
    .appName("BNGramDetection") \
    .config("spark.jars.packages", "com.johnsnowlabs.nlp:spark-nlp_2.12:5.5.2") \
    .getOrCreate()

    # Read the data with Spark
    tweetDF = spark.read.json(f"file://{temp_local_path}")
    print("Number of tweets loaded:", tweetDF.count())
    
    # Select relevant fields
    filteredDF = tweetDF.select(
        "created_at", "user.id", "user.name", "user.screen_name", "user.lang", "text"
    ).filter(col("text").isNotNull())
    print("Number of tweets before NLP-based filtering:", filteredDF.count())
    
    return filteredDF, spark

### Preprocessing Part 2 (The Pipeline)
Here we define the pipeline that we'll employ as our big preprocessing block.

The functions in our pipeline are the following:
- Document Assembler: Converts text into a format suitable for NLP.
- Language Detector: Identifies the language of the text.
- Tokenizer & Normalizer: Splits text into words and normalizes them (e.g., lowercasing).
- StopWords Cleaner: Removes common, non-informative words.
- Word Embeddings: Maps words to vector representations using GloVe embeddings.
- NER Model & Converter: Detects named entities (e.g., people, locations) and converts them into structured chunks.

After getting some sub-optimal results we found that there were many irrelevant words included in our top bigrams, so we created acustom list of stop words with help from *Chat-GPT*. This is included in the `custom_stopwords` variable.

In [0]:
# Function to clean and process text using Spark NLP
def nlp_pipeline(filteredDF):
    # Clean text: Remove links, mentions, and special characters
    cleanedDF = filteredDF.withColumn(
        "text",
        regexp_replace(col("text"), r"http\S+", "")  # Remove links
    ).withColumn(
        "text",
        regexp_replace(col("text"), r"@\w+", "")  # Remove mentions
    ).withColumn(
        "text",
        regexp_replace(col("text"), r"[^\w\s]", "")  # Remove special characters
    )
    
    # Document Assembler
    document_assembler = DocumentAssembler() \
        .setInputCol("text") \
        .setOutputCol("document")
    
    # Language Detector
    language_detector = LanguageDetectorDL.pretrained() \
        .setInputCols(["document"]) \
        .setOutputCol("language")
    
    # Tokenizer
    tokenizer = Tokenizer() \
        .setInputCols(["document"]) \
        .setOutputCol("token")
    
    # Normalizer
    normalizer = Normalizer() \
        .setInputCols(["token"]) \
        .setOutputCol("normalized") \
        .setLowercase(True)
    
    # StopWords Cleaner
    custom_stopwords = [

    # Pronouns, articles, conjunctions, etc.
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your",
    "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she",
    "her", "hers", "herself", "it", "this", "these", "its", "itself", "they", "them", "their", "more", "than",
    "theirs", "themselves", "am", "is", "are", "was", "were", "be", "been",
    "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a",
    "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "no",
    "of", "at", "by", "for", "with", "about", "between", "into", "through", "what", "so", "how", "many",
    "during", "to", "from", "in", "on", "off", "over", "under", "not",

    # Adverbs removed for lacking topic relevance
    "again", "then", "here", "there", "once", "further",

    # Social media specific
    "lol", "omg", "pls", "ur", "u", "btw", "idk", "tbh", "smh", "fyi",

    # Slang and contractions
    "r", "wanna", "gonna", "gotta", "ain't", "ya", "tho", "cuz", "b4","rt", "im", "ive", "dont", "m", "yall", "also", "irl"
]

    stopwords_cleaner = StopWordsCleaner() \
        .setInputCols(["normalized"]) \
        .setOutputCol("cleaned_tokens") \
        .setCaseSensitive(False) \
        .setStopWords(custom_stopwords)
    
    # Word Embeddings Model
    word_embeddings = WordEmbeddingsModel.pretrained("glove_100d", "en") \
        .setInputCols(["document", "cleaned_tokens"]) \
        .setOutputCol("word_embeddings")
    
    # NER Model
    ner_model = NerDLModel.pretrained("ner_dl", "en") \
        .setInputCols(["document", "cleaned_tokens", "word_embeddings"]) \
        .setOutputCol("ner_tags")
    
    # NER Converter
    ner_converter = NerConverter() \
        .setInputCols(["document", "cleaned_tokens", "ner_tags"]) \
        .setOutputCol("ner_chunks")
    
    # Create pipeline
    nlp_pipeline = Pipeline(stages=[
        document_assembler,
        language_detector,
        tokenizer,
        normalizer,
        stopwords_cleaner,
        word_embeddings,
        ner_model,
        ner_converter
    ])
    
    # Fit and transform the data
    result = nlp_pipeline.fit(cleanedDF).transform(cleanedDF)
    
    return result


###Filtering language and tokens in English
The following filters tweets in English. First, it uses a pretrained language detection model to identify tweets in english, then applies a used-defined function to validate that all the tokens contain valid english characters using regular expression, in other words, correct patterns.

In [0]:
# Define UDF to check for English-only tweets
def tokens_english(tokens):
    return all(bool(re.match(r"^[A-Za-z0-9\s.,!?'-]+$", token)) for token in tokens)

tokens_english_udf = udf(tokens_english, BooleanType())   

# Function to filter English tweets
def filter_english_tweets(result):
    result = result.filter(F.array_contains(col("language.result"), "en"))

    result = result.filter(tokens_english_udf(col("cleaned_tokens.result")))
    
    return result


###The Time Window
To detect relevant bigrams for bngrams we had to take into account the timeslots of several tweets in order to observe the evolution they had in time. The logic was a little complicated for us at first but in the end we grasped it and structured it in a simple way in one function.

Our `group_time_window` function works by converting the created_at column into a timestamp format and filtering out rows with invalid timestamps. The tweets are then grouped into 5-minute windows that slide every minute. Within each time window, the function aggregates two key pieces of information: flattened lists of tokenized words (`cleaned_tokens.result`) and detected named entities (`ner_chunks.result`), preparing the data for further analysis of trends or patterns over time.

In [0]:
# Group the tweets by time window
def group_time_window(result):

    spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

    result = result.withColumn(
        "timestamp", to_timestamp("created_at", "EEE MMM dd HH:mm:ss Z yyyy")
    ).filter(col("timestamp").isNotNull())

    windowedTokensDF = result.groupBy(
        window("timestamp", "5 minutes", "1 minute")
    ).agg(
        F.flatten(F.collect_list("cleaned_tokens.result")).alias("flattened_tweets"),
        F.flatten(F.collect_list("ner_chunks.result")).alias("flattened_ner_chunks")
    )

    return windowedTokensDF

###Generating Ngrams
This function generates bigrams from our previosuly tokenized tweets (`flattened_tweets`) using Spark NLP's NGram transformer and counts their occurrences in each time window. The counts are sorted by window and frequency, providing insight into the most common bigrams. Additionally, it joins named entity data (`flattened_ner_chunks`) to the results for context, enabling analysis of word pairings alongside identified entities.

In [0]:
# Function to detect bigrams (n = 2)
def generate_ngrams(windowedTokensDF, N = 2):
    
    # Generate bigrams
    ngram = NGram(n=N, inputCol="flattened_tweets", outputCol="bigrams")
    ngramsDF = ngram.transform(windowedTokensDF)

    # Count bigrams in each window
    ngramsDF = ngramsDF.withColumn("bigram", explode(col("bigrams"))) \
        .groupBy("window", "bigram") \
        .count() \
        .orderBy("window", "count", ascending=[True, False])

    # Join the flattened_ner_chunks column from the original DataFrame
    ngramsDF = ngramsDF.join(windowedTokensDF.select("window", "flattened_ner_chunks"), on="window", how="left")

    return ngramsDF

###Frequency penalization
The next step consists of creating a function to compute the `DF-IDF score`, as described in the paper, to penalize bigrams that are popular over time. It begins by assigning a unique ID to each time window to maintain chronological order. For each bigram, the average frequency is calculated and the following formula is applied:

`df-idft = (count + 1) / (log(historical_avg + 1) + 1)`

This formula adjusts the bigram's importance by penalizing those that have appeared frequently across multiple time windows, favoring emerging terms instead. The resulting dataframe includes the original counts, historical average of the frequency, and the penalized DF-IDF scores, which highlight bigrams that are gaining popularity rather than those that are consistently present.

In [0]:
def penalization_df_idft(ngramsDF, N = 2):

    ngramsDF = ngramsDF.withColumn("window.start", col("window.start").cast("timestamp"))
    # Assing ID to each window
    ngramsDF = ngramsDF.withColumn("window_ID", F.row_number().over(Window.orderBy("window.start")))

    # Define the window to calculate the historical average freq over past windows
    past_windows = Window.partitionBy("bigram").orderBy("window_ID").rowsBetween(Window.unboundedPreceding, -1)

    # Average historical freq - is used for the following formula
    ngramsDF = ngramsDF.withColumn(
        "historical_avg", 
        F.coalesce(
            F.sum("count").over(past_windows) / F.row_number().over(Window.partitionBy("bigram").orderBy("window_ID")),
            lit(0)  # assign 0 to null values
        )
    )

    # Formula of the df-idf from the paper
    ngramsDF = ngramsDF.withColumn("df_idft", (col("count")+1)/(F.log(col("historical_avg")+1)+1))

    return ngramsDF

###Boosting
The `boosting` function processes our DataFrame (`bigramsDF`) to adjust scores for bigrams based on their association with named entities. It begins by splitting each bigram string into an array of words, facilitating comparisons at the word level. Using the `flattened_ner_chunks` (a combined list of named entities extracted from the `ner_chunk` column), the function identifies whether any words in the bigram match these entities. A new boolean column, `contains_entity`, records whether a match exists. The final step adjusts the bigram's score (`df_idft`) by applying a boost multiplier (`BOOST`) if the bigram contains an entity, with the results stored in the `final_score` column and ordered by `window_ID`.

The column `ner_chunk` originates from named entity recognition (NER) processes and contains extracted entities like names, locations, or organizations. These chunks are flattened into `flattened_ner_chunks`, a consolidated list of all entities across the data, enabling efficient comparisons with the bigram arrays. By incorporating this entity-based relevance, the function effectively prioritizes bigrams linked to significant entities, refining the overall scoring process.

In [0]:
def boosting(ngramsDF, BOOST = 1.5, N = 2):

    # Transform the ngram into an array to separate the words
    ngramsDF = ngramsDF.withColumn("bigrams_array", split(col("bigram"), " "))

    # Compare the entities with the bigrams, if the bigram contains an entity the result of the new column will be True
    ngramsDF = ngramsDF.withColumn("contains_entity", 
    F.expr("array_intersect(flattened_ner_chunks, bigrams_array) IS NOT NULL AND size(array_intersect(flattened_ner_chunks, bigrams_array)) >0"))

    # Obtain the final score applying the boost to the frequency penalization column
    ngramsDF = ngramsDF.withColumn("final_score", when(col("contains_entity") == True, col("df_idft") * lit(BOOST)).otherwise(col("df_idft"))).orderBy("window_ID")

    return ngramsDF

###Clustering 
The next step in the Bngrams method was creating and fitting the clusters for different ngrams. To do this we first created a similarity matrix.

The `similarity_matrix` function creates a similarity matrix for bigrams based on their frequency within the same window. It calculates pairwise similarity scores by comparing the counts of each bigram, resulting in a matrix where higher values indicate greater similarity.

The `clustering` function takes the similarity matrix and applies hierarchical clustering to group similar bigrams into clusters. It transforms the similarity matrix into a distance matrix, then performs linkage and clustering using a given threshold. The result is a DataFrame showing each bigram and its corresponding cluster, organizing bigrams based on their similarity.


In [0]:
def similarity_matrix(df):
    # Similarity matrix for ngrams within the same window
    ngrams = df["bigram"].tolist()
    counts = df["count"].tolist()
    n = len(ngrams)

    # Initialize similarity matrix
    similarity_mat = np.zeros((n,n))
    for i in range(n):
        for j in range(n):
            if i != j:
                similarity_mat[i,j] = min(counts[i], counts[j]) / max(counts[i],counts[j])

    return similarity_mat, ngrams

def clustering(similarity_mat, ngrams, thres):

    # Transform the similarity matrix into a distance matrix
    distance_mat = 1 - similarity_mat

    # Optimize it by condensation
    distance_condensed = squareform(distance_mat, checks = False)

    # Hierarchical clustering using the average for grouping the clusters (mean distance)
    z = linkage(distance_condensed, method = "average")

    # Assign clusters to ngrams according to the threshold, distance they must be in order to be able to  combine
    clusters = fcluster(z, t = thres, criterion = "distance")

    # Data Frame with the results
    results_clustering = pd.DataFrame({"bigram": ngrams, "cluster":clusters})

    return results_clustering


###Hierarchical Clustering
The `hierarchical_clustering` function processes the bigrams from our DataFrame by applying hierarchical clustering based on a similarity threshold. The function iterates through distinct windows, filtering bigrams within each window based on a minimum score. After computing a similarity matrix for each window, it performs clustering and appends the results. The clustered bigrams are combined into a final DataFrame, sorted by cluster score and window, providing a detailed overview of trending topics for each time period.

The final output organizes bigrams into clusters, showcasing their relevance and significance within specific time windows, and provides insights into the most impactful topics based on similarity and scoring.

In [0]:
def hierarchical_clustering(bigramsDF, thres = 0.5, min_score = 1.3):
    # Process each window
    all_windows = bigramsDF.select("window").distinct().collect()
    results = [] # list 

    for row in all_windows:
        # Select a  window
        window = row["window"]

        # It is an interval, the window has start and end
        start_window, end_window = window[0], window[1]

        # Filter current window ngrams
        window_data = bigramsDF.filter((col("window.start") >= start_window) & (col("window.end") <= end_window) &(col("final_score") >= min_score)).toPandas()
        
        if window_data.empty:
            continue  # Avoid window without information 

        # Similarity matrix
        similarity_mat, bigrams = similarity_matrix(window_data)

        # Clustering
        clust_ngrams = clustering(similarity_mat, bigrams, thres)

        
        # Append needed columns from original dataframe 
        clust_ngrams["window"] = [window] * len(clust_ngrams)
        clust_ngrams["final_score"] = window_data["final_score"].tolist()       

        # Append the window clusters in general results
        clust_ngrams_reset = clust_ngrams.reset_index(drop=True) 
        results.append(clust_ngrams_reset)  # Add to the list

    
    # Create a single data frame 
    final_results = pd.concat(results, ignore_index=True)
    
    # Order clusters by the maximum score of their ngrams
    final_results = (
        final_results.groupby(["window", "cluster"])
        .agg({"final_score": "max", "bigram": lambda x: list(x)})
        .reset_index()
        .sort_values(by=["window", "final_score"], ascending=[True, False])
    )
    
    return final_results

###Summary Function
Until now our functional process has been very segmented and organized and here, finally, we wrap it all up in the generate_trending_topics_report function.

In summation it processes and analyzes the bigrams from our dataset to generate a `final report` on trending topics. It begins by setting constants like BOOST, N, thres, and min_score. The function then fetches and processes the data using the fetch_and_process_data and nlp_pipeline functions we previously explained. After filtering English tweets, the bigrams are detected and scored using penalization_df_idft and boosting methods. Finally, hierarchical clustering organizes the bigrams into clusters based on similarity, producing a detailed report.

In [0]:
# Central function to generate the report
def generate_trending_topics_report(start_year, start_month, start_day, start_hour, start_minute, duration_minutes):
    # Constants
    BOOST = 1.5
    N = 2
    thres = 0.5
    min_score = 1.3

    # Transform the input values to the correct format (2 digits)
    start_day = str(start_day).zfill(2)
    start_month = str(start_month).zfill(2)
    start_hour = str(start_hour).zfill(2)
    start_minute = str(start_minute).zfill(2)

    # Fetch and process data
    filteredDF, spark = fetch_and_process_data(start_year, start_month, start_day, start_hour, start_minute, duration_minutes)
    
    # Clean and process text
    result = nlp_pipeline(filteredDF)
    
    # Filter English tweets
    result = filter_english_tweets(result)

    # Group by time window
    windowedTokensDF = group_time_window(result)
    
    # Detect bigrams
    bigramsDF = generate_ngrams(windowedTokensDF, N)  

    # Calculate df - idf
    bigrams_DF_IDF= penalization_df_idft(bigramsDF)
    
    # Final score - boost for entities
    bigrams_DF_IDF= boosting(bigrams_DF_IDF, BOOST, N)

    # Hierarchical clustering based on the distance 
    report = hierarchical_clustering(bigrams_DF_IDF, thres, min_score)

    return report


###Running Example
In this final block we extract a specific time slot of the dataset and run our whole program on it to receive the final report, containing the optimal bigrams for our trending topic study.

In [0]:
# Example
start_year, start_month, start_day, start_hour, start_minute = 2019, 8, 1, 2, 0
duration_minutes = 5
report = generate_trending_topics_report(start_year, start_month, start_day, start_hour, start_minute, duration_minutes)

# Display the report
print("Trending Topics Report by time slot:")
display(report)

Number of tweets loaded: 11206
Number of tweets before NLP-based filtering: 10138
ld_wiki_tatoeba_cnn_21 download started this may take some time.
Approximate size to download 7.1 MB
[OK!]
glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[OK!]
ner_dl download started this may take some time.
Approximate size to download 13.6 MB
[OK!]
Trending Topics Report by time slot:


,window,cluster,final_score,bigram
0,"(2019-08-01 07:56:00, 2019-08-01 08:01:00)",1,10.000000,"[biggest fans, fans week, thank via, nct dream..."
2,"(2019-08-01 07:56:00, 2019-08-01 08:01:00)",3,7.500000,"[kamala harris, th june, find out, business mo..."
1,"(2019-08-01 07:56:00, 2019-08-01 08:01:00)",2,3.000000,"[harris fucked, fucked up, up bgt, bgt parah, ..."
3,"(2019-08-01 07:57:00, 2019-08-01 08:02:00)",1,8.000000,"[happy birthday, biggest fans, fans week, than..."
5,"(2019-08-01 07:57:00, 2019-08-01 08:02:00)",3,4.000000,"[kamala harris, fucked up, prescription drugs,..."
4,"(2019-08-01 07:57:00, 2019-08-01 08:02:00)",2,3.000000,"[harris fucked, up bgt, bgt parah, parah sad, ..."
9,"(2019-08-01 07:58:00, 2019-08-01 08:03:00)",4,8.146888,"[happy birthday, biggest fans, fans week, than..."
8,"(2019-08-01 07:58:00, 2019-08-01 08:03:00)",3,4.980558,"[kamala harris, season x, august pm, like that..."
7,"(2019-08-01 07:58:00, 2019-08-01 08:03:00)",2,4.000000,"[fucked up, set up, prescription drugs, up tod..."
6,"(2019-08-01 07:58:00, 2019-08-01 08:03:00)",1,3.000000,"[harris fucked, up bgt, bgt parah, parah sad, ..."


# Conclusion 

### Our Work
The project aimed to analyze trending topics using Twitter data. After reading the referenced data, we decided to choose the NGram detection algorithm, as it has the best general results according to their conclusions. The steps that were described on the paper have been followed in this assignment, and throughout this process, there have been several challenges and insights.

In general, the NGram detection algorithm focuses on identifying frequent and relevant word pairs in text. It analyzes co-occurrences of terms within a specific context and computes their importance based on the frequency penalization and boosting. Understanding the steps and functionality of the algorithm was one of the most challenging aspects of the project, how each stage contributed to detecting trending topics effectively. It allowed us to gain a deeper understanding of the algorithm's potential. Another problems that we encountered were the understanding of some already-created functions and creating the clustering algorithm, which required effort and time.

We focused on bigrams (N = 2) to be able to understand better the process and the algorithm as it was the first time that we were working with it. In the preprocessing step the data was trasnformed to be prepared for a meaningful analysis. Following the guidelines and the article, we implemented a text-cleaning and Spark NLP pipeline to remove links, mentions and special characters, performed tokenization, normalization, transform into lowercase, stopword removal and Name Entity Recognition. These are essential to obtain a structured data. In addition, the tweets must be filter using language detection to obtain only English words. While the article provided a general guidance, some adaptations were needed for our dataset. Aggregation and stemming were avoided, although they were mentioned in the paper, their performance was worse so we only take into account the tokenization, which was really useful.

The next steps were creating the window for the analysis, and generating the bigrams, which was easier than we thought thanks to the discovery of the function NGram. The only important thing was to prepare correctly the data it needs, and then adjust the dataframe with the columns that were needed for following steps.

To obtain better results the bigrams should be grouped according to the timeslot and their importance and to do so, a metric score was needed. The score used in this assignment consists of a penalization calculated according to the frequency of the terms, benefitting those new topics that emerge as trends, and penalizing the old topics. The formula, DF-IDF, was given in the article. Moreover, the terms that contain a proper name should have a higher score, as they are important topics, so a multiplication to the previous score is done by the boosting factor. 

This data is prepared for the clustering, with the goal of grouping similar topics and identifying the main trends. It is crucial for summarizing the most relevant information, detecting emerging patterns and obtain groups of words with logic. They were order according to the maximum score of the bigrams that each cluster contains (hierarchichal clustering)

The model aims to be as general as possible, in order to be able to apply it to several datasets and enabling the adjustment of parameters to adapt it to different scenarios. We selected some values of variables, such as threshold or minimum score to strike a balance between efficiency and result accuracy, some were stated in the article and others were adapted. 

The role of the AI has been important, it helps us in certain parts of the project, specially, with the general structure and steps. It was a good basis to be able to develop the code and adjust it to our dataset. For example, as it was explained before, in the creation of the stopwords list. Also, it was useful for clustering, as usually we perfomed this with a function and not "manually". The AI provided us with the steps and helped us understand how to adjsut it to our project. Further more, it is very effective explaining programming errors and solving them.

### Results and Improvements
Each row shows a significant trend identified during the analysis. The performance could be improved, as some of the trending topics are not very meaningful or easy to understand, but we consider that we have obtained a good result. 

The results illustrate how topics evolve over time, some trends like "biggest fans" maintain their presence across several time slots while others disappear quickly. 

We can see how the algorithm detect as the main topics words related with the elections, from Kamala Harris and her healthcare controversies to Donald Trump and his racist ideas. However, this is mixed with personal topics such as "Happy Birthday".

Some limitations were observed, particularly in the detection of named entity. The NER process tended to classify any uppercase word as a proper noun, which was inaccurate. To mitigate this, we applied lowercasing before (as recommended on the paper), however, this reduced the ability of the function to identify the proper nouns. A more advanced NER model could improve the results.
In addition, the stopwords list could be more precise in order to obtain a better preprocessing and to filter correctly the words.

### Future Steps
According to the results and with the conclusions that we have obtained from this assignment, some future steps that we could do include improving the preprocessing to obtain more accurate and meaningful results. 

We would explore more ideas and options, using other score metrics to order the clusters and the ngrams, do a deepeer study of the values of the variables like the minimal score and try to use other N, not only use bigrams for the algorithm. 

In addition, we could try to validate the results and visualize them to better understand them and do a better analysis.

In conclusion, this project has been really interesting as it was closer to a real problem and we used a real dataset. The paper has really helped us giving us a robust base and the steps to follow. We have been able to obtain meaningful insights from the results of the exploration of trending topics and from the whole project.

Emma Rodríguez Hervás.